<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/Seed-VC_V2%E7%9B%B4%E6%8E%A5%E6%8D%A2%E8%A7%86%E9%A2%91%E9%9F%B3%E8%89%B2%E5%92%8C%E9%94%99%E5%AD%97.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 指定人物全台词换音色：保留原节奏和口型

此版本改用 **Seed-VC V2声音转换**，不再用TTS重新朗读。

- 原视频决定文字、停顿、语速、快慢和抑扬顿挫；
- 参考声音只提供新音色；
- 通过字幕编号指定需要换音色的人物，其他人物保持原声；
- 背景音乐和音效继续保留。

最终输出 `/content/video_new_voice.mp4`。


In [ ]:
# 步骤1：确认使用 Colab Pro 的 NVIDIA L4 GPU
import os, subprocess
assert os.path.exists('/content'), '请在Google Colab中运行。'
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True).strip()
print('检测到：', gpu)
assert 'L4' in gpu, f'当前不是L4：{gpu}'


In [ ]:
# 步骤2：安装免费字幕、音轨分离和Seed-VC V2
%cd /content
!python -m pip install -q -U uv huggingface_hub hf_xet pysrt faster-whisper demucs pydub

from pathlib import Path
import subprocess, urllib.request

seed = Path('/content/seed-vc')
if not seed.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Plachtaa/seed-vc.git',str(seed)], check=True)
else:
    subprocess.run(['git','-C',str(seed),'fetch','--depth','1','origin','main'], check=True)
    subprocess.run(['git','-C',str(seed),'reset','--hard','origin/main'], check=True)

seed_python = seed/'.venv/bin/python'
if not seed_python.exists():
    subprocess.run(['uv','venv','--python','3.10',str(seed/'.venv')], check=True)

packages = [
    'torch==2.4.0', 'torchaudio==2.4.0', 'numpy==1.26.4', 'scipy==1.13.1',
    'librosa==0.10.2', 'huggingface-hub>=0.28.1', 'hf-xet', 'munch==4.0.0',
    'einops==0.8.0', 'pydub==0.25.1', 'transformers==4.46.3',
    'soundfile==0.12.1', 'hydra-core==1.3.2', 'pyyaml', 'accelerate', 'tqdm', 'pysrt'
]
subprocess.run(['uv','pip','install','--python',str(seed_python),*packages], check=True)

base = 'https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/'
urllib.request.urlretrieve(base+'auto_replace_voice.py', seed/'auto_replace_voice.py')
urllib.request.urlretrieve(base+'generate_srt.py', '/content/generate_srt.py')
urllib.request.urlretrieve(base+'prepare_audio.py', '/content/prepare_audio.py')
print('✅ 安装完成；Seed-VC模型会在步骤5首次运行时自动下载。')


## 步骤3：上传视频和新音色参考声音

同时选择：①原视频；②想要替换成的新音色参考声音。参考声音建议清晰、无音乐、5～20秒。


In [ ]:
# 步骤3：上传、生成字幕并分离音轨
from google.colab import files
from pathlib import Path
import subprocess, sys, pysrt

uploaded = files.upload()
names = list(uploaded)
def one(exts, label):
    matches = [n for n in names if Path(n).suffix.lower() in exts]
    if len(matches) != 1:
        raise ValueError(f'需要且只能上传一个{label}；识别到：{matches}')
    return matches[0]

video = one({'.mp4','.mov','.mkv','.webm'}, '视频')
voice = one({'.wav','.mp3','.m4a','.flac','.ogg'}, '参考声音')
Path('/content/video.mp4').write_bytes(uploaded[video])
voice_source = Path('/content/uploaded_voice'+Path(voice).suffix.lower())
voice_source.write_bytes(uploaded[voice])
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(voice_source),'-ac','1','-ar','24000','/content/voice.wav'], check=True)

subprocess.run([sys.executable,'/content/generate_srt.py','--video','/content/video.mp4','--output','/content/subtitles.srt','--model','large-v3','--language','zh'], check=True)
subprocess.run([sys.executable,'/content/prepare_audio.py'], check=True)

print('✅ 字幕如下，请记住需要换音色人物的编号：')
for row in pysrt.open('/content/subtitles.srt', encoding='utf-8'):
    print(f'#{row.index}  {row.start} --> {row.end}  |  {row.text.replace(chr(10), " " )}')


## 步骤4：填写这个人物的全部字幕编号

例如 `1-4,7,9-12` 表示选择第1至4、7、9至12句。只选择同一个人物说的台词。


In [ ]:
TARGET_LINES = ''  # 示例：'1-4,7,9-12'

import json, re, pysrt
from pathlib import Path
if not str(TARGET_LINES).strip():
    raise ValueError('请填写TARGET_LINES。')
Path('/content/voice_conversion_config.json').write_text(
    json.dumps({'target_lines':TARGET_LINES}, ensure_ascii=False, indent=2), encoding='utf-8'
)

def parse_spec(value):
    text = str(value).replace('，', ',').replace('—', '-').replace('–', '-')
    result = []
    for part in filter(None, (x.strip() for x in text.split(','))):
        if '-' in part:
            a, b = map(int, part.split('-', 1)); result.extend(range(min(a,b), max(a,b)+1))
        else:
            result.append(int(part))
    return sorted(set(result))

subs = {row.index:row for row in pysrt.open('/content/subtitles.srt', encoding='utf-8')}
print('✅ 将更换这些台词的音色：')
for number in parse_spec(TARGET_LINES):
    if number not in subs: raise ValueError(f'字幕编号不存在：{number}')
    print(f'#{number} | {subs[number].text.replace(chr(10), " " )}')


In [ ]:
# 步骤5：把选中人物全部台词转换成参考音色
import os, subprocess, urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/auto_replace_voice.py','/content/seed-vc/auto_replace_voice.py')
env = os.environ.copy()
env['PYTHONPATH'] = '/content/seed-vc' + os.pathsep + env.get('PYTHONPATH','')
subprocess.run(['/content/seed-vc/.venv/bin/python','/content/seed-vc/auto_replace_voice.py'], cwd='/content/seed-vc', env=env, check=True)
print('✅ 输出：/content/video_new_voice.mp4')


In [ ]:
# 步骤6：预览并下载成品
from IPython.display import Video, display
from google.colab import files
output = '/content/video_new_voice.mp4'
display(Video(output, embed=True))
files.download(output)
